In [3]:
from __future__ import annotations

from pathlib import Path
from uuid import uuid4
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import polars as pl

from src.data import IngestionAgent
from src.deployment import AdversarialStressTester, DeploymentHarness
from src.evaluation import EvaluationEngine
from src.features import PurgedEraSplitter
from src.models import ModelOrchestrator
from src.notebook_utils import render_promotion_report
from src.risk import NeutralizationEngine
from src.runner import PromotionRunner

agent = IngestionAgent(project_root / 'data' / 'v5.2')
target_col = agent.get_target_names('train')[0]
custom_engine = EvaluationEngine(
    era_col='era',
    prediction_col='prediction',
    target_col=target_col,
    id_col='id',
    backend='custom',
)
official_engine = EvaluationEngine(
    era_col='era',
    prediction_col='prediction',
    target_col=target_col,
    id_col='id',
    backend='official',
)
neutralizer = NeutralizationEngine(project_root / 'artifacts' / 'cache')
harness = DeploymentHarness()

In [4]:
# Research and smoke-check scratchpad.
# Keep ad hoc EDA, anchor-fold trials, and feature experiments in this cell.

train_lazy = agent.scan_dataset('train', include_metadata=True)
print('Train schema columns:', len(train_lazy.collect_schema().names()))
print('Available feature subsets:', agent.feature_subset_names())

Train schema columns: 3
Available feature subsets: ('small', 'medium', 'all', 'v2_equivalent_features', 'v3_equivalent_features', 'fncv3_features', 'intelligence', 'charisma', 'strength', 'dexterity', 'constitution', 'wisdom', 'agility', 'serenity', 'sunshine', 'rain', 'midnight', 'faith')


In [5]:
# ---------------------------------------------------------
# PROMOTION RUN CONFIGURATION
# ---------------------------------------------------------
PROMOTION_CONFIG = {
    'dataset': 'train',
    'feature_subset': 'small',
    'cv_splits': 5,
    'cv_purge_buffer': 1,
    'era_limit': 30,
    'model_params': {
        'n_estimators': 120,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_child_samples': 20,
    },
    'gates': {
        'min_mean_corr': -1.0,
        'min_sharpe_corr': -10.0,
        'max_drawdown_corr': 10.0,
    },
    'stress_testing': {
        'noise_std_ratio': 0.05,
        'degradation_threshold': 0.40,
    },
    'parity_eras': 5,
}

In [6]:
selected_eras = (
    agent.scan_dataset(PROMOTION_CONFIG['dataset'], include_metadata=True)
    .select('era')
    .unique()
    .sort('era')
    .collect()
    .get_column('era')
    .to_list()
)
if PROMOTION_CONFIG['era_limit'] is not None:
    selected_eras = selected_eras[:PROMOTION_CONFIG['era_limit']]

runner = PromotionRunner(
    agent=agent,
    dataset_name=PROMOTION_CONFIG['dataset'],
    feature_subset=PROMOTION_CONFIG['feature_subset'],
    target_column=target_col,
    splitter=PurgedEraSplitter(
        n_splits=PROMOTION_CONFIG['cv_splits'],
        purge_buffer=PROMOTION_CONFIG['cv_purge_buffer'],
    ),
    orchestrator=ModelOrchestrator(
        feature_names=agent.get_feature_names(PROMOTION_CONFIG['feature_subset']),
        target_column=target_col,
        model_library='lightgbm',
        model_params=PROMOTION_CONFIG['model_params'],
        prefer_gpu=True,
    ),
    custom_evaluation_engine=custom_engine,
    official_evaluation_engine=official_engine,
    neutralization_engine=neutralizer,
    stress_tester=AdversarialStressTester(
        custom_engine,
        degradation_threshold=PROMOTION_CONFIG['stress_testing']['degradation_threshold'],
        random_state=7,
    ),
    deployment_harness=harness,
    artifact_dir=project_root / 'artifacts' / 'promotion' / f"run_{uuid4().hex[:8]}" ,
    config_metadata=PROMOTION_CONFIG,
    gate_kwargs=PROMOTION_CONFIG['gates'],
    requirements_path=project_root / 'requirements.txt',
    era_slice=selected_eras,
    parity_era_count=PROMOTION_CONFIG['parity_eras'],
    stress_noise_std_ratio=PROMOTION_CONFIG['stress_testing']['noise_std_ratio'],
)

result = runner.run()
render_promotion_report(result)

c:\dev\numer-AI-refactored\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\dev\numer-AI-refactored\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\dev\numer-AI-refactored\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\dev\numer-AI-refactored\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\dev\numer-AI-refactored\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\dev\numer-AI-refactored\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\dev\num

### Promotion Run: PASS
---
#### 1. Out-of-Fold Evaluation (Custom Backend)
- **Mean CORR**: `0.02194`
- **Sharpe**: `0.854`
- **Max Drawdown**: `0.0298`
- **Mean FNC**: `0.01723`
- **Max Feature Exposure**: `0.43196`
#### 2. Institutional Gates
- **Fast-Fail Gate**: `PASS`
- **Oracle Parity** (Eras: 0003, 0013, 0019, 0021, 0030): `PASS`
- **Stress Test Degradation**: `21.51%`
#### 3. Deployment Artifact
- **Path**: `C:\dev\numer-AI-refactored\artifacts\promotion\run_c51d4520`
- **Smoke Test**: `PASS (Exact Match)`
---
<details><summary><b>Execution Logs (Expand)</b></summary>

```text
[1/8] Ingest & Split
Loaded dataset='train' subset='small' rows=80403 eras=30
[2/8] Train
Cross-validation trained models=5 backend=lightgbm-gpu fit_seconds=4.452
[3/8] Evaluate (Custom)
Custom evaluation mean_corr=0.021937 mean_fnc=0.017228 max_feature_exposure=0.431961
[4/8] Gate
Fast-fail gate passed
[5/8] Oracle Parity (Live Slice)
Oracle parity passed for eras=0003,0013,0019,0021,0030 | mean_corr=0.025507 mean_fnc=0.023828
[6/8] Stress Test (v1)
Stress test passed degradation_pct=21.514
[7/8] Serialize
Wrote artifact bundle to C:\dev\numer-AI-refactored\artifacts\promotion\run_c51d4520
Recorded SHA-256 hashes for model files: model_01.txt=842e7710b37039afca7c495ec45445a6fd26f406c4626436b4e99efd27781c0f, model_02.txt=842711f94d46c2fdfa08a5a038bd3a5c8cc7650998731912478d73246d71f854, model_03.txt=1f6f956e813402385d294ba819fe64d258ec2b267a69182a72122a7e61a5f304, model_04.txt=974ee47787431672687a71d49ffa1dcc4ae27115f6c4e82d67e23c795f2b93c1, model_05.txt=5d59377a5bb0991d5b06aee297f59eaf0d47d274d6c4c25482beb47b47b23496
[8/8] Smoke Test
Post-reload smoke test passed with exact prediction match
```

</details>